In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# lunar lander

# ppo from scratch

In [ ]:
# 1. Install System Dependency 'swig' (Required for Box2D)
!sudo apt-get update
!sudo apt-get install -y swig

# 2. Install Gymnasium with Box2D
!pip install "gymnasium[box2d]"
!pip install "stable-baselines3" "moviepy" "shimmy"

print("✅ Box2D Installed. PLEASE RESTART SESSION NOW.")

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical, Normal
import numpy as np
import gymnasium as gym

# Hyperparameters
lr = 0.0003
gamma = 0.99
gae_lambda = 0.95
clip_epsilon = 0.2
update_epochs = 10
batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, is_continuous, action_std_init=0.6):
        super(ActorCritic, self).__init__()
        self.is_continuous = is_continuous
        self.action_dim = action_dim
        
        # Shared Layers
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        
        # Actor Head
        self.actor = nn.Linear(64, action_dim)
        
        # Critic Head
        self.critic = nn.Linear(64, 1)
        
        # Continuous Action Specific: Learnable Log Standard Deviation
        if is_continuous:
            self.action_var = nn.Parameter(torch.full((action_dim,), action_std_init * action_std_init))

    def forward(self):
        raise NotImplementedError
    
    def act(self, state):
        x = torch.tanh(self.fc1(state))
        x = torch.tanh(self.fc2(x))
        
        action_mean = self.actor(x)
        value = self.critic(x)
        
        if self.is_continuous:
            # Continuous: Sample from Normal Distribution
            action_var = self.action_var.expand_as(action_mean)
            cov_mat = torch.diag_embed(action_var).to(device)
            dist = Normal(action_mean, action_var.sqrt()) # simplified diagonal
            # Usually for bounded actions we use Tanh, but simple Normal works for learning
        else:
            # Discrete: Sample from Categorical (Softmax)
            probs = F.softmax(action_mean, dim=-1)
            dist = Categorical(probs)

        action = dist.sample()
        action_logprob = dist.log_prob(action)
        
        if self.is_continuous:
            # Sum log probs for multi-dimensional continuous actions
            action_logprob = torch.sum(action_logprob, dim=-1)
            
        return action.detach(), action_logprob.detach(), value.detach()
    
    def evaluate(self, state, action):
        x = torch.tanh(self.fc1(state))
        x = torch.tanh(self.fc2(x))
        
        action_mean = self.actor(x)
        value = self.critic(x)
        
        if self.is_continuous:
            action_var = self.action_var.expand_as(action_mean)
            dist = Normal(action_mean, action_var.sqrt())
            # For multi-dim continuous, log_prob is summed
            action_logprobs = torch.sum(dist.log_prob(action), dim=-1)
            dist_entropy = torch.sum(dist.entropy(), dim=-1)
        else:
            probs = F.softmax(action_mean, dim=-1)
            dist = Categorical(probs)
            action_logprobs = dist.log_prob(action)
            dist_entropy = dist.entropy()
            
        return action_logprobs, value, dist_entropy

class PPO:
    def __init__(self, state_dim, action_dim, is_continuous):
        self.policy = ActorCritic(state_dim, action_dim, is_continuous).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.policy_old = ActorCritic(state_dim, action_dim, is_continuous).to(device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        
        self.MseLoss = nn.MSELoss()

    def update(self, memory):
        # Convert list to tensor
        rewards = []
        discounted_reward = 0
        for reward, is_terminal in zip(reversed(memory.rewards), reversed(memory.is_terminals)):
            if is_terminal:
                discounted_reward = 0
            discounted_reward = reward + (gamma * discounted_reward)
            rewards.insert(0, discounted_reward)
            
        # Normalizing the rewards
        rewards = torch.tensor(rewards, dtype=torch.float32).to(device)
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-7)
        
        # Convert memory to tensors
        old_states = torch.squeeze(torch.stack(memory.states, dim=0)).detach().to(device)
        old_actions = torch.squeeze(torch.stack(memory.actions, dim=0)).detach().to(device)
        old_logprobs = torch.squeeze(torch.stack(memory.logprobs, dim=0)).detach().to(device)
        
        # Optimize policy for K epochs
        for _ in range(update_epochs):
            # Evaluating old actions and values
            logprobs, state_values, dist_entropy = self.policy.evaluate(old_states, old_actions)
            
            # Finding the Ratio (pi_theta / pi_theta__old)
            ratios = torch.exp(logprobs - old_logprobs)

            # Finding Surrogate Loss
            advantages = rewards - state_values.detach().squeeze()
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1-clip_epsilon, 1+clip_epsilon) * advantages
            
            # Final Loss = -min(surr1, surr2) + 0.5*MSE(val, reward) - 0.01*Entropy
            loss = -torch.min(surr1, surr2) + 0.5 * self.MseLoss(state_values.squeeze(), rewards) - 0.01 * dist_entropy
            
            # take gradient step
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()
            
        # Copy new weights into old policy
        self.policy_old.load_state_dict(self.policy.state_dict())
        
class Memory:
    def __init__(self):
        self.actions = []
        self.states = []
        self.logprobs = []
        self.rewards = []
        self.is_terminals = []
    
    def clear_memory(self):
        del self.actions[:]
        del self.states[:]
        del self.logprobs[:]
        del self.rewards[:]
        del self.is_terminals[:]

print("✅ PPO Engine Ready.")

✅ PPO Engine Ready.


In [4]:
import gymnasium as gym
import torch
import numpy as np

# --- CONFIGURATION ---
env_name = "LunarLander-v2"
env = gym.make(env_name)

# 1. Detect Environment Spaces
state_dim = env.observation_space.shape[0]
# LunarLander is Discrete: action_dim is an integer (4 actions)
action_dim = env.action_space.n 
is_continuous = False 

# 2. Initialize Custom PPO Agent
# We pass is_continuous=False so it uses Categorical distribution (Softmax)
ppo_agent = PPO(state_dim, action_dim, is_continuous)
memory = Memory()

print(f"🚀 Initializing Mars Rover Mission on {env_name}...")
print(f"   State Dimensions: {state_dim}")
print(f"   Action Dimensions: {action_dim} (Discrete: Main/Side Engines)")

# 3. Training Hyperparameters
max_episodes = 1500       # Increased to ensure convergence
log_interval = 20         # Print average reward every 20 episodes
update_timestep = 2000    # Update PPO policy every 2000 timesteps
time_step = 0

running_reward = 0
avg_length = 0

# 4. Training Loop
print("--> Mission Start (Training)...")

for i_episode in range(1, max_episodes+1):
    state, _ = env.reset()
    current_ep_reward = 0
    
    for t in range(1000): # Max timesteps per episode
        time_step += 1
        
        # A. Interact with Environment
        # The agent decides an action based on the current state
        state_tensor = torch.FloatTensor(state).to(device)
        action, logprob, _ = ppo_agent.policy_old.act(state_tensor)
        
        # For discrete environments, action.item() gives the integer index (0,1,2,3)
        state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        
        # B. Store Data in Memory
        memory.states.append(state_tensor)
        memory.actions.append(action)
        memory.logprobs.append(logprob)
        memory.rewards.append(reward)
        memory.is_terminals.append(done)
        
        # C. PPO Update (Learning)
        # When enough data is collected, update the neural network
        if time_step % update_timestep == 0:
            ppo_agent.update(memory)
            memory.clear_memory()
            time_step = 0
            
        current_ep_reward += reward
        if done:
            break
            
    # Logging
    running_reward += current_ep_reward
    avg_length += t
    
    if i_episode % log_interval == 0:
        avg_length = int(avg_length/log_interval)
        running_reward = int((running_reward/log_interval))
        
        print(f"Episode {i_episode} \t Avg Reward: {running_reward} \t Avg Length: {avg_length}")
        
        # Save if solved, but keep training to stabilize
        if running_reward > 200:
            print("✅ MISSION ACCOMPLISHED! Rover is landing consistently.")
            torch.save(ppo_agent.policy.state_dict(), 'ppo_rover_scratch.pth')
            
        running_reward = 0
        avg_length = 0

# --- FORCE SAVE AT THE END ---
print("💾 Saving final model state...")
torch.save(ppo_agent.policy.state_dict(), 'ppo_rover_scratch.pth')
print("🏁 Training Finished. Model saved as 'ppo_rover_scratch.pth'.")

🚀 Initializing Mars Rover Mission on LunarLander-v2...
   State Dimensions: 8
   Action Dimensions: 4 (Discrete: Main/Side Engines)
--> Mission Start (Training)...
Episode 20 	 Avg Reward: -173 	 Avg Length: 104
Episode 40 	 Avg Reward: -158 	 Avg Length: 105
Episode 60 	 Avg Reward: -178 	 Avg Length: 92
Episode 80 	 Avg Reward: -146 	 Avg Length: 97
Episode 100 	 Avg Reward: -140 	 Avg Length: 92
Episode 120 	 Avg Reward: -121 	 Avg Length: 97
Episode 140 	 Avg Reward: -127 	 Avg Length: 91
Episode 160 	 Avg Reward: -156 	 Avg Length: 98
Episode 180 	 Avg Reward: -113 	 Avg Length: 89
Episode 200 	 Avg Reward: -141 	 Avg Length: 98
Episode 220 	 Avg Reward: -114 	 Avg Length: 103
Episode 240 	 Avg Reward: -117 	 Avg Length: 96
Episode 260 	 Avg Reward: -140 	 Avg Length: 108
Episode 280 	 Avg Reward: -114 	 Avg Length: 98
Episode 300 	 Avg Reward: -146 	 Avg Length: 108
Episode 320 	 Avg Reward: -130 	 Avg Length: 94
Episode 340 	 Avg Reward: -100 	 Avg Length: 104
Episode 360 	 Avg 

In [5]:
import glob
import io
import base64
import torch
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML, display

# 1. Define the Video Display Function
def show_video(video_dir):
    mp4list = glob.glob(f'{video_dir}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay 
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else: 
        print("Could not find video")

# 2. Setup Environment for Recording
print("🎥 Setting up Recording Studio...")
env = gym.make("LunarLander-v2", render_mode="rgb_array")
env = RecordVideo(env, video_folder="rover_video", episode_trigger=lambda x: True)

# 3. Load Your Trained Model
# We need to re-initialize the architecture to load the weights
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
agent = ActorCritic(state_dim, action_dim, is_continuous=False).to(device)

# Load the weights you just trained
try:
    agent.load_state_dict(torch.load('ppo_rover_scratch.pth'))
    print("✅ Loaded trained model: ppo_rover_scratch.pth")
except:
    print("⚠️ Error: Model not found. Did the training finish successfully?")

# 4. Play ONE Episode
state, _ = env.reset()
done = False
total_reward = 0

print("🚀 Rover is attempting to land...")
while not done:
    # Get action from trained agent
    state_tensor = torch.FloatTensor(state).to(device)
    action, _, _ = agent.act(state_tensor)
    
    # Step environment
    state, reward, terminated, truncated, _ = env.step(action.item())
    total_reward += reward
    done = terminated or truncated

env.close()

# 5. Show the Result
print(f"🏁 Final Score: {total_reward:.2f}")
if total_reward > 200:
    print("🏆 Result: PERFECT LANDING")
else:
    print("💥 Result: CRASH or ROUGH LANDING")

print("\n👇 YOUR VIDEO IS BELOW 👇")
show_video('rover_video')

🎥 Setting up Recording Studio...
✅ Loaded trained model: ppo_rover_scratch.pth
🚀 Rover is attempting to land...


/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /kaggle/working/rover_video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.11/dist-packages/gymnasium/core.py:297: UserWarning: WARN: env.is_vector_env to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.is_vector_env` for environment variables or `env.get_attr('is_vector_env')` that will search the reminding wrappers.
  logger.warn(


Moviepy - Building video /kaggle/working/rover_video/rl-video-episode-0.mp4.
Moviepy - Writing video /kaggle/working/rover_video/rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready /kaggle/working/rover_video/rl-video-episode-0.mp4
🏁 Final Score: 28.60
💥 Result: CRASH or ROUGH LANDING

👇 YOUR VIDEO IS BELOW 👇


In [6]:
import gymnasium as gym
import torch
import numpy as np

# --- CONFIGURATION ---
env_name = "BipedalWalker-v3"
env = gym.make(env_name)

# 1. Detect Environment Spaces
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0] # Continuous: 4 Dimensions
is_continuous = True                   # <--- KEY DIFFERENCE

# 2. Initialize Custom PPO Agent
# We pass is_continuous=True so it uses Normal Distribution (Gaussian)
ppo_agent = PPO(state_dim, action_dim, is_continuous)
memory = Memory()

print(f"🤖 Initializing Robotics Simulation on {env_name}...")
print(f"   State Dimensions: {state_dim}")
print(f"   Action Dimensions: {action_dim} (Continuous Joint Torques)")

# 3. Training Hyperparameters
max_episodes = 2000       # Walker needs more practice
log_interval = 20         
update_timestep = 4000    # More data needed per update for continuous
time_step = 0

running_reward = 0
avg_length = 0

# 4. Training Loop
print("--> Simulation Start (Training)...")

for i_episode in range(1, max_episodes+1):
    state, _ = env.reset()
    current_ep_reward = 0
    
    # Walker episodes can be long, we cap at 1600
    for t in range(1600): 
        time_step += 1
        
        # A. Interact with Environment
        state_tensor = torch.FloatTensor(state).to(device)
        action, logprob, _ = ppo_agent.policy_old.act(state_tensor)
        
        # For continuous, we pass the numpy array directly
        # action is a Tensor [4], env.step expects Array [4]
        state, reward, terminated, truncated, _ = env.step(action.cpu().numpy())
        done = terminated or truncated
        
        # B. Store Data
        memory.states.append(state_tensor)
        memory.actions.append(action)
        memory.logprobs.append(logprob)
        memory.rewards.append(reward)
        memory.is_terminals.append(done)
        
        # C. PPO Update
        if time_step % update_timestep == 0:
            ppo_agent.update(memory)
            memory.clear_memory()
            time_step = 0
            
        current_ep_reward += reward
        if done:
            break
            
    # Logging & Saving
    running_reward += current_ep_reward
    avg_length += t
    
    if i_episode % log_interval == 0:
        avg_length = int(avg_length/log_interval)
        running_reward = int((running_reward/log_interval))
        
        print(f"Episode {i_episode} \t Avg Reward: {running_reward} \t Avg Length: {avg_length}")
        
        # Walker is "Solved" at 300 points
        if running_reward > 300:
            print("✅ ROBOT IS WALKING! Saving model...")
            torch.save(ppo_agent.policy.state_dict(), 'ppo_walker_scratch.pth')
            break
            
        running_reward = 0
        avg_length = 0

# --- FORCE SAVE ---
print("💾 Saving final walker brain...")
torch.save(ppo_agent.policy.state_dict(), 'ppo_walker_scratch.pth')
print("🏁 Training Finished.")

🤖 Initializing Robotics Simulation on BipedalWalker-v3...
   State Dimensions: 24
   Action Dimensions: 4 (Continuous Joint Torques)
--> Simulation Start (Training)...
Episode 20 	 Avg Reward: -100 	 Avg Length: 468
Episode 40 	 Avg Reward: -99 	 Avg Length: 536
Episode 60 	 Avg Reward: -92 	 Avg Length: 916
Episode 80 	 Avg Reward: -94 	 Avg Length: 761
Episode 100 	 Avg Reward: -96 	 Avg Length: 692
Episode 120 	 Avg Reward: -94 	 Avg Length: 914
Episode 140 	 Avg Reward: -69 	 Avg Length: 1446
Episode 160 	 Avg Reward: -78 	 Avg Length: 1104
Episode 180 	 Avg Reward: -70 	 Avg Length: 1218
Episode 200 	 Avg Reward: -62 	 Avg Length: 1395
Episode 220 	 Avg Reward: -73 	 Avg Length: 1137
Episode 240 	 Avg Reward: -54 	 Avg Length: 1458
Episode 260 	 Avg Reward: -63 	 Avg Length: 1335
Episode 280 	 Avg Reward: -48 	 Avg Length: 1447
Episode 300 	 Avg Reward: -32 	 Avg Length: 1523
Episode 320 	 Avg Reward: -17 	 Avg Length: 1523
Episode 340 	 Avg Reward: 8 	 Avg Length: 1448
Episode 36

In [8]:
import glob
import io
import base64
import torch
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML, display

# Helper function to display video
def show_video(video_dir):
    mp4list = glob.glob(f'{video_dir}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay 
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else: 
        print(f"No video found in {video_dir}")

# Define Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. RECORD "BEFORE" (RANDOM AGENT)
# ==========================================
print("🎥 Recording BEFORE Training (Random)...")
env_random = gym.make("BipedalWalker-v3", render_mode="rgb_array")
env_random = RecordVideo(env_random, video_folder="walker_video_before", episode_trigger=lambda x: True)

state, _ = env_random.reset()
done = False
total_reward_random = 0

while not done:
    action = env_random.action_space.sample() # Random Action
    state, reward, terminated, truncated, _ = env_random.step(action)
    total_reward_random += reward
    done = terminated or truncated

env_random.close()
print(f"💥 Random Score: {total_reward_random:.2f}")


# ==========================================
# 2. RECORD "AFTER" (PPO AGENT)
# ==========================================
print("\n🎥 Recording AFTER Training (PPO)...")
env_ppo = gym.make("BipedalWalker-v3", render_mode="rgb_array")
env_ppo = RecordVideo(env_ppo, video_folder="walker_video_after", episode_trigger=lambda x: True)

# Load Model Structure
state_dim = env_ppo.observation_space.shape[0]
action_dim = env_ppo.action_space.shape[0]
agent = ActorCritic(state_dim, action_dim, is_continuous=True).to(device)

# Load Weights
try:
    agent.load_state_dict(torch.load('ppo_walker_scratch.pth'))
    print("✅ Loaded trained weights: ppo_walker_scratch.pth")
except:
    print("⚠️ Error: Model weights not found. Did training finish?")

# Run Episode
state, _ = env_ppo.reset()
done = False
total_reward_ppo = 0

while not done:
    state_tensor = torch.FloatTensor(state).to(device)
    action, _, _ = agent.act(state_tensor)
    
    state, reward, terminated, truncated, _ = env_ppo.step(action.cpu().numpy())
    total_reward_ppo += reward
    done = terminated or truncated

env_ppo.close()
print(f"🏆 Trained Score: {total_reward_ppo:.2f}")


# ==========================================
# 3. DISPLAY RESULTS
# ==========================================
print("\n" + "="*40)
print("🔴 VIDEO 1: BEFORE TRAINING (Random)")
print("="*40)
show_video('walker_video_before')

print("\n" + "="*40)
print("🟢 VIDEO 2: AFTER PPO TRAINING")
print("="*40)
show_video('walker_video_after')

🎥 Recording BEFORE Training (Random)...


/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /kaggle/working/walker_video_before folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Moviepy - Building video /kaggle/working/walker_video_before/rl-video-episode-0.mp4.
Moviepy - Writing video /kaggle/working/walker_video_before/rl-video-episode-0.mp4



/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /kaggle/working/walker_video_after folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Moviepy - Done !
Moviepy - video ready /kaggle/working/walker_video_before/rl-video-episode-0.mp4
💥 Random Score: -99.27

🎥 Recording AFTER Training (PPO)...
✅ Loaded trained weights: ppo_walker_scratch.pth
Moviepy - Building video /kaggle/working/walker_video_after/rl-video-episode-0.mp4.
Moviepy - Writing video /kaggle/working/walker_video_after/rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready /kaggle/working/walker_video_after/rl-video-episode-0.mp4
🏆 Trained Score: -37.37

🔴 VIDEO 1: BEFORE TRAINING (Random)



🟢 VIDEO 2: AFTER PPO TRAINING
